In [5]:
import re
from typing import Dict, List
import datetime
import csv

DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [6]:
def validar_producto(codigo: str) -> Dict:
    resultado = {"valido": False, "categoria": None, "numero": None, "pais": None}
    patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
    match = re.match(patron, codigo)
    if match:
        resultado["valido"]    = True
        resultado["categoria"] = match.group(1)
        resultado["numero"]    = match.group(2)
        resultado["pais"]      = match.group(3)
    return resultado
    

In [7]:
def validar_envio(codigo: str) -> Dict:
    resultado = {"valido": False, "fecha": None, "secuencial": None}
    patron = r'^ENV-(202[0-9]|2030)-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])-(\d{6})$'
    match = re.match(patron, codigo)
    if match:
        anio, mes, dia, seq = match.group(1), match.group(2), match.group(3), match.group(4)
        resultado["valido"]     = True
        resultado["fecha"]      = f"{anio}-{mes}-{dia}"
        resultado["secuencial"] = seq
    return resultado

In [8]:
def validar_empleado(codigo: str) -> Dict:
    resultado = {"valido": False, "departamento": None, "numero": None}
    patron = r'^EMP-([A-Z]{3})-([1-9]\d{3})$'
    match = re.match(patron, codigo)
    if match:
        depto  = match.group(1)
        numero = match.group(2)
        if depto in DEPARTAMENTOS_VALIDOS:
            resultado["valido"]       = True
            resultado["departamento"] = depto
            resultado["numero"]       = numero
    return resultado

In [9]:
def validar_factura(codigo: str) -> Dict:
    resultado = {"valido": False, "serie": None, "numero": None}
    patron = r'^FAC-([A-E])-(\d{6})$'
    match = re.match(patron, codigo)
    if match:
        resultado["valido"]  = True
        resultado["serie"]   = match.group(1)
        resultado["numero"]  = match.group(2)
    return resultado

In [10]:
def validar_codigo(codigo: str) -> Dict:
    resultado = {
        "codigo":   codigo,
        "tipo":     "desconocido",
        "valido":   False,
        "detalles": {}
    }
    if codigo.startswith("ENV"):
        tipo = "envio"
        detalles = validar_envio(codigo)
    elif codigo.startswith("EMP"):
        tipo = "empleado"
        detalles = validar_empleado(codigo)
    elif codigo.startswith("FAC"):
        tipo = "factura"
        detalles = validar_factura(codigo)
    elif re.match(r'^[A-Za-z]{3}-', codigo):
        tipo = "producto"
        detalles = validar_producto(codigo)
    else:
        return resultado

    resultado["tipo"]     = tipo
    resultado["valido"]   = detalles["valido"]
    resultado["detalles"] = {k: v for k, v in detalles.items() if k != "valido"}
    return resultado

In [11]:
def procesar_lote(codigos: List[str]) -> Dict:
    resultado = {
        "total": 0, "validos": 0, "invalidos": 0,
        "por_tipo": {
            "producto":    {"total": 0, "validos": 0},
            "envio":       {"total": 0, "validos": 0},
            "empleado":    {"total": 0, "validos": 0},
            "factura":     {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0},
        },
        "detalle": []
    }
    for codigo in codigos:
        res = validar_codigo(codigo)
        tipo = res["tipo"]
        resultado["total"] += 1
        resultado["por_tipo"][tipo]["total"] += 1
        if res["valido"]:
            resultado["validos"] += 1
            resultado["por_tipo"][tipo]["validos"] += 1
        else:
            resultado["invalidos"] += 1
        resultado["detalle"].append(res)
    return resultado

In [12]:
def mostrar_resultado(resultado: Dict) -> None:
    estado = "✓" if resultado["valido"] else "✗"
    print(f"{estado} {resultado['codigo']:<30} | Tipo: {resultado['tipo']:<12}")
    if resultado["valido"] and resultado["detalles"]:
        detalles = ", ".join(f"{k}: {v}" for k, v in resultado["detalles"].items() if v)
        print(f"   └── {detalles}")

def mostrar_reporte(reporte: Dict) -> None:
    print("=" * 60)
    print("                 REPORTE DE VALIDACIÓN")
    print("=" * 60)
    print(f"\nTotal procesados: {reporte['total']}")
    print(f"Válidos:   {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
    print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")
    print("\nDesglose por tipo:")
    print("-" * 40)
    for tipo, stats in reporte["por_tipo"].items():
        if stats["total"] > 0:
            tasa = stats["validos"] / stats["total"] * 100
            print(f"  {tipo.capitalize():<12}: {stats['validos']:>3}/{stats['total']:<3} ({tasa:.0f}% válidos)")
    print("\n" + "=" * 60)

In [13]:
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

print("\n-- Productos --")
print(validar_producto("TEC-0001-MX"))
print(validar_producto("tec-0001-MX"))

print("\n-- Envíos --")
print(validar_envio("ENV-2024-03-15-001234"))
print(validar_envio("ENV-2024-13-15-001234"))

print("\n-- Empleados --")
print(validar_empleado("EMP-VEN-1234"))
print(validar_empleado("EMP-VEN-0123"))

print("\n-- Facturas --")
print(validar_factura("FAC-A-123456"))
print(validar_factura("FAC-F-123456"))

PRUEBA DE FUNCIONES INDIVIDUALES

-- Productos --
{'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}

-- Envíos --
{'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}
{'valido': False, 'fecha': None, 'secuencial': None}

-- Empleados --
{'valido': True, 'departamento': 'VEN', 'numero': '1234'}
{'valido': False, 'departamento': None, 'numero': None}

-- Facturas --
{'valido': True, 'serie': 'A', 'numero': '123456'}
{'valido': False, 'serie': None, 'numero': None}


In [14]:
CODIGOS_PRUEBA = [
    "TEC-0001-MX", "ALI-9999-US", "ROB-1234-CA",
    "tec-0001-MX", "TEC-001-MX", "TECH-0001-MX",
    "ENV-2024-03-15-001234", "ENV-2025-12-01-999999",
    "ENV-2019-03-15-001234", "ENV-2024-13-15-001234", "ENV-2024-03-32-001234",
    "EMP-VEN-1234", "EMP-TEC-9999", "EMP-ADM-1000",
    "EMP-VEN-0123", "EMP-XXX-1234", "EMP-VEN-123",
    "FAC-A-123456", "FAC-E-000001", "FAC-B-999999",
    "FAC-F-123456", "FAC-A-12345", "FAC-a-123456",
    "XXX-1234", "RANDOM-CODE",
]

print("PRUEBA DE VALIDADOR UNIVERSAL")
print("=" * 50)
for codigo in CODIGOS_PRUEBA[:10]:
    mostrar_resultado(validar_codigo(codigo))

PRUEBA DE VALIDADOR UNIVERSAL
✓ TEC-0001-MX                    | Tipo: producto    
   └── categoria: TEC, numero: 0001, pais: MX
✓ ALI-9999-US                    | Tipo: producto    
   └── categoria: ALI, numero: 9999, pais: US
✓ ROB-1234-CA                    | Tipo: producto    
   └── categoria: ROB, numero: 1234, pais: CA
✗ tec-0001-MX                    | Tipo: producto    
✗ TEC-001-MX                     | Tipo: producto    
✗ TECH-0001-MX                   | Tipo: desconocido 
✓ ENV-2024-03-15-001234          | Tipo: envio       
   └── fecha: 2024-03-15, secuencial: 001234
✓ ENV-2025-12-01-999999          | Tipo: envio       
   └── fecha: 2025-12-01, secuencial: 999999
✗ ENV-2019-03-15-001234          | Tipo: envio       
✗ ENV-2024-13-15-001234          | Tipo: envio       


In [ ]:
reporte = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte)

In [18]:
def sugerir_correccion(codigo: str) -> str:
    candidato = codigo.upper()
    res = validar_codigo(candidato)
    if res["valido"]:
        return f"¿Quisiste decir: {candidato}?"
    return "No se encontró una corrección automática."

def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    try:
        datetime.date(anio, mes, dia)
        return True
    except ValueError:
        return False

def exportar_resultados(reporte: Dict, archivo: str) -> None:
    with open(archivo, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["codigo", "tipo", "valido", "detalles"])
        writer.writeheader()
        for item in reporte["detalle"]:
            writer.writerow({
                "codigo":   item["codigo"],
                "tipo":     item["tipo"],
                "valido":   item["valido"],
                "detalles": str(item["detalles"])
            })
    print(f"✓ Resultados exportados a 'resultados.csv'")

print(sugerir_correccion("tec-0001-MX"))
print("¿31 feb válido?", validar_fecha_real(2024, 2, 31))
print("¿15 mar válido?", validar_fecha_real(2024, 3, 15))
# Ensure reporte exists (run cell 11 first, or regenerate here if needed)
if 'reporte' not in locals():
    reporte = procesar_lote(CODIGOS_PRUEBA)

exportar_resultados(reporte, "resultados.csv")

¿Quisiste decir: TEC-0001-MX?
¿31 feb válido? False
¿15 mar válido? True
✓ Resultados exportados a 'resultados.csv'
